# LabOne Q — Workflow Exercise: e-f Rabi, the Applications Library way

This exercise follows `04_Workflows.ipynb`. In `02_Case_Study_ef_Rabi.ipynb` the e-f Rabi was written in the plain
DSL: pulses defined by hand, `ExperimentSignal`s mapped by hand, the g-e/e-f oscillator switch on the shared
port fixed by hand. Here the same experiment is rebuilt **exactly the way every experiment in
`laboneq_applications` is written**:

- **qubits and quantum operations** instead of pulses and signal names — the `QPU` carries the parameters
- a `create_experiment` task decorated with `@dsl.qubit_experiment`, so the experiment signals come from the qubits
- an `experiment_workflow` with the standard signature `(session, qpu, qubits, *, <sweep points>, temporary_parameters, options)`
- the shipped `compile_experiment` / `run_experiment` tasks, and a conditional `analysis_workflow` branch

The reference is `laboneq_applications/experiments/amplitude_rabi.py` — an e-f Rabi is that experiment with
`transition="ef"`. Two cells below are left for you to fill in, marked with `...`.

Everything runs in **emulation mode** (`do_emulation=True`) against a real SHFQC description, so the real
compiler is active.

In [ ]:
from __future__ import annotations

import numpy as np

from laboneq.simple import *
from laboneq import workflow
from laboneq.workflow.tasks import compile_experiment, run_experiment
from laboneq.dsl.quantum import QPU
from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation

from laboneq_applications.qpu_types.tunable_transmon import TunableTransmonQubit, TunableTransmonOperations
from laboneq_applications.experiments.options import TuneupExperimentOptions, TuneUpWorkflowOptions
from laboneq_applications.core import validation
from laboneq_applications.tasks import temporary_qpu, temporary_quantum_elements_from_qpu
from laboneq_applications.analysis.amplitude_rabi import analysis_workflow

## Setup

Same SHFQC as in the case study, `drive` and `drive_ef` on one physical port. The signal names follow the
Applications Library convention (`q0/drive`, `q0/drive_ef`, `q0/measure`, `q0/acquire`) — this is what lets
`TunableTransmonQubit.from_device_setup` build the qubit from the setup in the next step.

In [ ]:
setup = DeviceSetup(uid="workflow_exercise_setup")
setup.add_dataserver(host="localhost", port=8004)

setup.add_instruments(
    SHFQC(uid="device_shfqc", address="DEV12345", device_options="SHFQC/16W/LRT/PLUS/QC6CH/RTR", interface="1GBe")
)

setup.add_connections(
    "device_shfqc",
    create_connection(to_signal="q0/drive",    ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/drive_ef", ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/measure",  ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/acquire",  ports="QACHANNELS/0/INPUT"),
)

## QPU instead of a hand-written `Calibration`

In the case study every LO, IF and range was typed into a `Calibration` object. With a qubit, the same
numbers are **qubit parameters** — and `q0.calibration()` generates the `Calibration` from them. The `QPU`
bundles the qubit with the `TunableTransmonOperations` (`prepare_state`, `x180`, `measure`, ... from Lecture 2).

In [ ]:
[q0] = TunableTransmonQubit.from_device_setup(setup)

q0.update(
    drive_lo_frequency=4.0e9,
    resonance_frequency_ge=4.1e9,    # g-e IF: +100 MHz
    resonance_frequency_ef=3.9e9,    # e-f IF: -100 MHz, same LO, same port
    readout_lo_frequency=7.0e9,
    readout_resonator_frequency=7.08e9,
    ge_drive_amplitude_pi=0.7,
    ge_drive_length=100e-9,
    ef_drive_length=100e-9,
    readout_amplitude=0.5,
    readout_length=2e-6,
    readout_integration_length=2e-6,
)

qpu = QPU(quantum_elements=[q0], quantum_operations=TunableTransmonOperations())
setup.set_calibration(q0.calibration())

session = Session(device_setup=setup)
session.connect(do_emulation=True)

## Exercise 1 — `create_experiment`

This is the shape of every `create_experiment` in the library:

- `@workflow.task` on the outside, `@dsl.qubit_experiment` on the inside — the experiment signals are taken
  from the qubits passed in, no `ExperimentSignal` / `map_to` anywhere
- an `options` object (`TuneupExperimentOptions`: `count`, `averaging_mode`, `acquisition_type`, `cal_states`, ...)
- `validation.validate_and_convert_qubits_sweeps` normalises "one qubit + one array" and "list of qubits +
  list of arrays" to the same thing
- one `SweepParameter` per qubit, then the real-time loop, then calibration traces after the sweep

**Fill in `main_drive`**: bring the qubit to `e`, then play the e-f π-pulse with the swept amplitude — two
quantum operations. Compare with the case study: no `on_system_grid` fix and no second section. The operations
build the sections, and the compiler schedules the oscillator switch on the shared port — check it in the
pulse sheet further down.

In [ ]:
@workflow.task
@dsl.qubit_experiment
def create_experiment(qpu, qubits, amplitudes, options: TuneupExperimentOptions | None = None):
    opts = TuneupExperimentOptions() if options is None else options
    qubits, amplitudes = validation.validate_and_convert_qubits_sweeps(qubits, amplitudes)
    [q], [q_amplitudes] = qubits, amplitudes
    qop = qpu.quantum_operations

    amplitude = SweepParameter(f"ef_amplitude_{q.uid}", q_amplitudes, axis_name=q.uid)

    with dsl.acquire_loop_rt(
        count=opts.count,
        averaging_mode=opts.averaging_mode,
        acquisition_type=opts.acquisition_type,
    ):
        with dsl.sweep(name="ef_rabi_amp_sweep", parameter=amplitude):
            with dsl.section(name="main_drive", alignment=SectionAlignment.RIGHT):
                ...
            with dsl.section(name="main_measure"):
                qop.measure(q, dsl.handles.result_handle(q.uid))
                qop.passive_reset(q)

        qop.calibration_traces.omit_section(qubits=qubits, states=opts.cal_states)

## Exercise 2 — `experiment_workflow`

The signature is the library's, unchanged: `session, qpu, qubits` positional, sweep points and
`temporary_parameters` keyword-only, `options` last. **Fill in the body**, in this order:

1. `temporary_qpu(qpu, temporary_parameters)` — a copy of the QPU with the overrides applied
2. `temporary_quantum_elements_from_qpu(temp_qpu, qubits)` — UIDs → qubit objects, from that copy
3. `create_experiment(...)`, `compile_experiment(...)`, `run_experiment(...)`
4. `with workflow.if_(options.do_analysis):` → `analysis_workflow(result, qubits, amplitudes)`
5. `workflow.return_(result)`

Remember nothing runs at call time — you are describing a graph. `if` has to be `workflow.if_`.

In [ ]:
@workflow.workflow(name="ef_rabi")
def experiment_workflow(
    session,
    qpu,
    qubits,
    *,
    amplitudes,
    temporary_parameters=None,
    options: TuneUpWorkflowOptions | None = None,
):
    ...

## Run it

`experiment_workflow.options()` collects the options of every task in the graph into one object: `count` goes
to `create_experiment`, `do_analysis` to the workflow itself. The analysis is switched off here — emulated data
has no Rabi oscillation to fit — but the branch is still visible in the tree.

In [ ]:
options = experiment_workflow.options()
options.count(10)
options.do_analysis(False)

wf = experiment_workflow(session, qpu, "q0", amplitudes=np.linspace(0.0, 1.0, 11), options=options)
print(wf.graph.tree)

In [ ]:
wf_result = wf.run()
print([t.name for t in wf_result.tasks])

compiled_exp = wf_result.tasks["compile_experiment"].output
plot_simulation(compiled_exp, length=compiled_exp.estimated_runtime, signal_names_to_show=["drive", "drive_ef", "measure"])

The pulse sheet shows what the two operations built: a `prepare_state` section holding the g-e π-pulse and an
`x180` section holding the e-f pulse, one after the other and packed against the readout by the right-aligned
`main_drive`, then the calibration traces after the sweep.

In [ ]:
show_pulse_sheet("ef_rabi_workflow", compiled_exp, interactive=True)

## `temporary_parameters`

Because the workflow goes through `temporary_qpu`, it accepts the same overrides as every library workflow —
here a different g-e π-amplitude for the state preparation, without touching `qpu`.

In [ ]:
wf_result_override = experiment_workflow(
    session, qpu, "q0",
    amplitudes=np.linspace(0.0, 1.0, 11),
    temporary_parameters={"q0": {"ge_drive_amplitude_pi": 0.3}},
    options=options,
).run()

print("qpu untouched:", qpu["q0"].parameters.ge_drive_amplitude_pi)

## The same thing, shipped

What you just wrote is `amplitude_rabi.experiment_workflow` with `transition="ef"`. Same tree, same tasks — the
library version additionally supports several qubits in parallel, active reset, and the `evaluate` / `update`
branches that Lecture 4 builds on.

In [ ]:
from laboneq_applications.experiments import amplitude_rabi

lib_options = amplitude_rabi.experiment_workflow.options()
lib_options.transition("ef")
lib_options.count(10)
lib_options.do_analysis(False)

lib_wf = amplitude_rabi.experiment_workflow(session, qpu, "q0", amplitudes=np.linspace(0.0, 1.0, 11), options=lib_options)
print(lib_wf.graph.tree)

lib_compiled = lib_wf.run().tasks["compile_experiment"].output
plot_simulation(lib_compiled, length=lib_compiled.estimated_runtime, signal_names_to_show=["drive", "drive_ef", "measure"])